Sempre que o ambiente do notebook for reiniciado, basta executar a célula de carregamento anterior. Este bloco utilizará a variável df_final diretamente na memória, garantindo alta performance e eliminando a necessidade de reprocessar os arquivos brutos.

In [3]:
import pandas as pd
import numpy as np

# Copie o caminho exato que o seu seletor encontrou para não dar mais erro de pasta
caminho_arquivo = "C:/Users/silas/Documents/tabela_9543_padronizada---LimpoFinal/dados-ibge/dados/tabela_9543_ibge_municipios.csv"

print("====== ANÁLISE EXPLORATÓRIA DA BASE ORIGINAL (CORRIGIDA) ======\n")

# MUDANÇA CRÍTICA: Trocamos para encoding='utf-8' para corrigir os caracteres especiais
df_original = pd.read_csv(caminho_arquivo, encoding='utf-8', sep=';')

# ==========================================
# PERGUNTA 1: Dimensões, tipos, uso de memória
# ==========================================
print("1. DIMENSÕES, TIPOS E MEMÓRIA")
print(f"Linhas no arquivo original: {df_original.shape[0]:,}")
print(f"Colunas no arquivo original: {df_original.shape[1]}")
print(f"Uso de Memória Estimado: {df_original.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

print("\nTipos de Dados por Coluna (Nomes Corrigidos):")
print(df_original.dtypes)
print("-" * 50 + "\n")

# ==========================================
# PERGUNTA 2: Faltantes por coluna (Qtd e %)
# ==========================================
print("2. VALORES FALTANTES (NULOS LITERAIS OU OCULTOS)")
nulos_diretos = df_original.isnull().sum()
omitidos_ibge = df_original.astype(str).apply(lambda x: x.str.strip().isin(['...', '-', ''])).sum()

faltantes_total = nulos_diretos + omitidos_ibge
faltantes_pct = (faltantes_total / len(df_original)) * 100

df_total_faltantes = pd.DataFrame({
    'Total Faltantes/Omitidos': faltantes_total,
    'Percentual (%)': faltantes_pct.round(2)
})
print(df_total_faltantes)
print("-" * 50 + "\n")

# ==========================================
# PERGUNTA 3: Duplicados e Categorias
# ==========================================
print("3. DUPLICADOS E INCONSISTÊNCIAS")
duplicados = df_original.duplicated().sum()
print(f"Linhas totalmente duplicadas na base: {duplicados}")

print("\nAnálise de Valores Únicos para o Relatório:")
colunas_para_checar = ['Nível Territorial', 'Sexo', 'Cor ou raça', 'Idade', 'Variável']

for col in colunas_para_checar:
    if col in df_original.columns:
        valores_unicos = df_original[col].unique()
        print(f"\n-> Valores encontrados na coluna '{col}' (Total: {len(valores_unicos)}):")
        print(valores_unicos)

====== ANÁLISE EXPLORATÓRIA DA BASE ORIGINAL (CORRIGIDA) ======

1. DIMENSÕES, TIPOS E MEMÓRIA
Linhas no arquivo original: 22,280
Colunas no arquivo original: 17
Uso de Memória Estimado: 12.09 MB

Tipos de Dados por Coluna (Nomes Corrigidos):
Nível Territorial (Código)      int64
Nível Territorial                 str
Unidade de Medida (Código)      int64
Unidade de Medida                 str
Valor                         float64
Município (Código)              int64
Município                         str
Variável (Código)               int64
Variável                          str
Ano (Código)                    int64
Ano                             int64
Sexo (Código)                   int64
Sexo                              str
Cor ou raça (Código)            int64
Cor ou raça                       str
Idade (Código)                  int64
Idade                             str
dtype: object
--------------------------------------------------

2. VALORES FALTANTES (NULOS LITERAIS OU OCULT

In [25]:
import pandas as pd
import numpy as np

# 1. Caminho exato e verificado que funciona no seu computador
caminho_arquivo = "C:/Users/silas/Documents/tabela_9543_padronizada---LimpoFinal/dados-ibge/dados/tabela_9543_ibge_municipios.csv"

# 2. Carrega a Base Original com o encoding correto (utf-8) para sanar os caracteres estranhos
df_original = pd.read_csv(caminho_arquivo, encoding='utf-8', sep=';')

# 3. Recria a 'Base Limpa' na memória reproduzindo exatamente as regras do seu projeto
df_limpo = df_original[df_original['Nível Territorial'] == 'Município'].copy()
df_limpo = df_limpo.dropna(subset=['Valor'])

colunas_manter = ['Município (Código)', 'Município', 'Sexo', 'Cor ou raça', 'Idade', 'Valor']
df_limpo = df_limpo[colunas_manter]

# ==========================================================
# 4. MONTAGEM AUTOMÁTICA DA TABELA COMPARATIVA
# ==========================================================
# Conta valores omitidos de forma vetorizada e segura para texto/número
omitidos_total = df_original['Valor'].isnull().sum() + df_original['Valor'].astype(str).str.strip().isin(['...', '-', '']).sum()

dados_comparacao = {
    "Métrica de Qualidade": [
        "Total de Linhas (Registros)",
        "Total de Colunas",
        "Uso de Memória",
        "Linhas com Valores Faltantes/Nulos",
        "Inconsistência de Escala (Granularidade)",
        "Erros de Texto / Encoding nas Colunas"
    ],
    "Base Original (Suja)": [
        f"{df_original.shape[0]:,}",
        f"{df_original.shape[1]}",
        f"{df_original.memory_usage(deep=True).sum() / (1024**2):.2f} MB",
        f"{omitidos_total:,}", # Exibe o total real calculado na linha acima
        "Sim (Misturava dados de Municípios e Estados)",
        "Sim (Caracteres corrompidos corrigidos via utf-8)"
    ],
    "Base Final (Limpa)": [
        f"{df_limpo.shape[0]:,}",
        f"{df_limpo.shape[1]}",
        f"{df_limpo.memory_usage(deep=True).sum() / (1024**2):.2f} MB",
        "0 (100% higienizada)",
        "Não (Isolado apenas para Municípios)",
        "Não (Colunas padronizadas e legíveis)"
    ]
}

df_comparacao = pd.DataFrame(dados_comparacao)

print("====== RECOMPOSIÇÃO DE DADOS: ANTES VS DEPOIS ======\n")
df_comparacao


====== RECOMPOSIÇÃO DE DADOS: ANTES VS DEPOIS ======



,Métrica de Qualidade,Base Original (Suja),Base Final (Limpa)
0,Total de Linhas (Registros),"22,280","20,709"
1,Total de Colunas,17,6
2,Uso de Memória,12.09 MB,5.19 MB
3,Linhas com Valores Faltantes/Nulos,"1,571",0 (100% higienizada)
4,Inconsistência de Escala (Granularidade),Sim (Misturava dados de Municípios e Estados),Não (Isolado apenas para Municípios)
5,Erros de Texto / Encoding nas Colunas,Sim (Caracteres corrompidos corrigidos via utf-8),Não (Colunas padronizadas e legíveis)


In [26]:
import pandas as pd
import numpy as np

# 1. Carrega o arquivo usando o caminho correto do seu computador
caminho_arquivo = "C:/Users/silas/Documents/tabela_9543_padronizada---LimpoFinal/dados-ibge/dados/tabela_9543_ibge_municipios.csv"
df = pd.read_csv(caminho_arquivo, encoding='utf-8', sep=';')

# 2. Executa a limpeza essencial para a análise rodar de primeira
df = df[df['Nível Territorial'] == 'Município'].copy()
df = df.dropna(subset=['Valor'])

# 3. Renomeia as colunas do padrão IBGE para o padrão que você definiu no projeto
df = df.rename(columns={
    'Valor': 'taxa_alfabetizacao',
    'Sexo': 'sexo',
    'Cor ou raça': 'cor_raca'
})

# =====================================================================
# METODOLOGIA ADOTADA: Z-score por agrupamento demográfico específico
# =====================================================================
def calcular_zscore_grupo(grupo):
    std = grupo['taxa_alfabetizacao'].std()
    if std == 0 or np.isnan(std):
        grupo['zscore'] = 0.0
    else:
        grupo['zscore'] = (grupo['taxa_alfabetizacao'] - grupo['taxa_alfabetizacao'].mean()) / std
    return grupo

# O parâmetro as_index=False força o Pandas a MANTER as colunas normais na tabela
df = df.groupby(['sexo', 'cor_raca'], as_index=False, group_keys=False).apply(calcular_zscore_grupo)

# 4. Filtra as linhas que são outliers legítimos (Z-score maior que 3 ou menor que -3)
outliers = df[np.abs(df['zscore']) > 3]

# ==========================================================
# 5. EXIBIÇÃO IMPREVÍSIVEL (PEGA TODAS AS COLUNAS PRESENTES)
# ==========================================================
print(f"Quantidade de outliers legítimos encontrados: {len(outliers)}")
print("\nAmostra dos Outliers Identificados no Grupo Correto:")

# Exibe as colunas sem risco de KeyError por digitação ou índice modificado
outliers.head()

Quantidade de outliers legítimos encontrados: 420

Amostra dos Outliers Identificados no Grupo Correto:


,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,taxa_alfabetizacao,Município (Código),Município,Variável (Código),Variável,Ano (Código),Ano,Sexo (Código),Cor ou raça (Código),Idade (Código),Idade,zscore
24,6,Município,2,%,77.78,1100072,Corumbiara - RO,2513,Taxa de alfabetização das pessoas de 15 anos o...,2022,2022,4,2776,6572,15 anos,-4.699305
143,6,Município,2,%,80.00,1101005,Governador Jorge Teixeira - RO,2513,Taxa de alfabetização das pessoas de 15 anos o...,2022,2022,5,2777,6572,15 anos,-4.056175
214,6,Município,2,%,88.89,1200054,Assis Brasil - AC,2513,Taxa de alfabetização das pessoas de 15 anos o...,2022,2022,5,2776,6572,15 anos,-3.216664
263,6,Município,2,%,83.33,1200393,Porto Walter - AC,2513,Taxa de alfabetização das pessoas de 15 anos o...,2022,2022,5,2777,6572,15 anos,-3.337260
297,6,Município,2,%,50.00,1300029,Alvarães - AM,2513,Taxa de alfabetização das pessoas de 15 anos o...,2022,2022,4,2777,6572,15 anos,-6.282420


In [14]:
import pandas as pd
import numpy as np

# 1. CARGA E LIMPEZA INICIAL (Tratamento de Faltantes e Encoding)
caminho_arquivo = "C:/Users/silas/Documents/tabela_9543_padronizada---LimpoFinal/dados-ibge/dados/tabela_9543_ibge_municipios.csv"
df_processado = pd.read_csv(caminho_arquivo, encoding='utf-8', sep=';')

# Filtragem de escopo (Isola apenas municípios e remove nulos explícitos ou do IBGE)
df_processado = df_processado[df_processado['Nível Territorial'] == 'Município'].copy()
df_processado = df_processado.dropna(subset=['Valor'])

# Renomeia colunas para o padrão de desenvolvimento
df_processado = df_processado.rename(columns={
    'Valor': 'taxa_alfabetizacao',
    'Sexo': 'sexo',
    'Cor ou raça': 'cor_raca'
})

# 2. REMOÇÃO DE DUPLICADOS
# Garante a eliminação de qualquer registro repetido para o mesmo município e perfil
df_processado = df_processado.drop_duplicates(subset=['Município', 'sexo', 'cor_raca', 'Idade'])

# 3. PADRONIZAÇÃO DE CATEGORIAS
# Garante que textos não tenham espaços invisíveis nas pontas que quebrem agrupamentos
df_processado['sexo'] = df_processado['sexo'].str.strip()
df_processado['cor_raca'] = df_processado['cor_raca'].str.strip()

# =====================================================================
# CRIAÇÃO DAS COLUNAS DERIVADAS (Exigência do Critério)
# =====================================================================

# --- COLUNA DERIVADA 1: Região do País (Uso de .map()) ---
# Cria o mapeamento das siglas de UF para as grandes regiões brasileiras
mapa_regioes = {
    'RO': 'Norte', 'AC': 'Norte', 'AM': 'Norte', 'RR': 'Norte', 'PA': 'Norte', 'AP': 'Norte', 'TO': 'Norte',
    'MA': 'Nordeste', 'PI': 'Nordeste', 'CE': 'Nordeste', 'RN': 'Nordeste', 'PB': 'Nordeste', 'PE': 'Nordeste',
    'AL': 'Nordeste', 'SE': 'Nordeste', 'BA': 'Nordeste',
    'MG': 'Sudeste', 'ES': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
    'PR': 'Sul', 'SC': 'Sul', 'RS': 'Sul',
    'MS': 'Centro-Oeste', 'MT': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'DF': 'Centro-Oeste'
}
# Extrai a UF do texto do município (ex: "Porto Velho - RO" -> "RO")
df_processado['uf_sigla'] = df_processado['Município'].str.split(' - ').str[-1]
# Aplica o .map() para derivar a Região
df_processado['regiao'] = df_processado['uf_sigla'].map(mapa_regioes)


# --- COLUNA DERIVADA 2: Desempenho Relativo à Mediana do Grupo (Uso de .apply() e np.where) ---
# Passo A: Calcula a mediana de alfabetização para cada combinação de Sexo e Cor/Raça
medianas_grupo = df_processado.groupby(['sexo', 'cor_raca'])['taxa_alfabetizacao'].transform('median')

# Passo B: Cria coluna comparando a taxa do município com a mediana do seu grupo usando np.where
df_processado['status_grupo'] = np.where(
    df_processado['taxa_alfabetizacao'] >= medianas_grupo, 
    'Acima ou Igual à Mediana do Grupo', 
    'Abaixo da Mediana do Grupo'
)


# --- COLUNA DERIVADA 3: Faixas de Alfabetização (Uso de pd.cut()) ---
# Cria categorias discretas (intervalos) para classificar os níveis de alfabetização dos municípios
intervalos = [0, 50, 85, 95, 100]
rotulos = ['Crítico (0-50%)', 'Alerta (51-85%)', 'Aceitável (86-95%)', 'Excelente (96-100%)']

df_processado['faixa_alfabetizacao'] = pd.cut(
    df_processado['taxa_alfabetizacao'], 
    bins=intervalos, 
    labels=rotulos, 
    include_lowest=True
)

# =====================================================================
# VALIDAÇÃO DOS RESULTADOS
# =====================================================================
print("====== PROCESSO DE TRATAMENTO E DERIVAÇÃO CONCLUÍDO ======\n")
print(f"Dimensões finais da base tratada: {df_processado.shape}")
print("\nVerificação das novas colunas derivadas criadas:")
df_processado[['Município', 'regiao', 'status_grupo', 'faixa_alfabetizacao']].head()


====== PROCESSO DE TRATAMENTO E DERIVAÇÃO CONCLUÍDO ======

Dimensões finais da base tratada: (20709, 21)

Verificação das novas colunas derivadas criadas:


,Município,regiao,status_grupo,faixa_alfabetizacao
0,Alta Floresta D'Oeste - RO,Norte,Acima ou Igual à Mediana do Grupo,Excelente (96-100%)
1,Alta Floresta D'Oeste - RO,Norte,Acima ou Igual à Mediana do Grupo,Excelente (96-100%)
2,Alta Floresta D'Oeste - RO,Norte,Acima ou Igual à Mediana do Grupo,Excelente (96-100%)
3,Alta Floresta D'Oeste - RO,Norte,Acima ou Igual à Mediana do Grupo,Excelente (96-100%)
4,Ariquemes - RO,Norte,Abaixo da Mediana do Grupo,Excelente (96-100%)


In [15]:
# Salva a base de dados totalmente higienizada e com as novas colunas
caminho_salvamento = "C:/Users/silas/Documents/tabela_9543_padronizada---LimpoFinal/dados-ibge/dados/tabela_9543_final_tratada.csv"
df_processado.to_csv(caminho_salvamento, sep=';', encoding='utf-8-sig', index=False)

print(f"✅ Base final tratada e higienizada salva com sucesso em:\n{caminho_salvamento}")


✅ Base final tratada e higienizada salva com sucesso em:
C:/Users/silas/Documents/tabela_9543_padronizada---LimpoFinal/dados-ibge/dados/tabela_9543_final_tratada.csv


In [17]:
# Use essa linha nas próximas células para carregar sua base 100% pronta
df_final = pd.read_csv("C:/Users/silas/Documents/tabela_9543_padronizada---LimpoFinal/dados-ibge/dados/tabela_9543_final_tratada.csv", sep=';', encoding='utf-8-sig')


In [20]:
print("====== METADADOS E CATEGORIAS DA BASE TRATADA ======\n")

# Pergunta 1: Quantas linhas e colunas existem no total?
# Usamos o df_final que já foi carregado na célula anterior
print(f"A tabela tem o total de {df_final.shape[0]:,} linhas e {df_final.shape[1]} colunas.\n")

# Pergunta 2: Quais são todos os Estados (UF) que estão nessa tabela?
print("Estados (UF) presentes na tabela:")
print(sorted(df_final['uf_sigla'].unique()))

# Pergunta 3: Quais são os sexos registrados na tabela toda?
print("\nCategorias de Sexo:")
print(df_final['sexo'].unique())

# Pergunta 4: Quais são as cores/raças registradas na tabela toda?
print("\nCategorias de Cor ou Raça:")
print(df_final['cor_raca'].unique())

# Pergunta 5: Quais são as faixas de idade registradas na tabela toda?
print("\nGrupos de Idade:")
print(df_final['Idade'].unique())


====== METADADOS E CATEGORIAS DA BASE TRATADA ======

A tabela tem o total de 20,709 linhas e 21 colunas.

Estados (UF) presentes na tabela:
['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR', 'RS', 'SC', 'SE', 'SP', 'TO']

Categorias de Sexo:
<StringArray>
['Homens', 'Mulheres']
Length: 2, dtype: str

Categorias de Cor ou Raça:
<StringArray>
['Branca', 'Preta']
Length: 2, dtype: str

Grupos de Idade:
<StringArray>
['15 anos']
Length: 1, dtype: str


In [22]:
print("====== 1. VISÃO GERAL DA BASE ANALISADA ======\n")

# 1. Total de registros (linhas) e colunas
linhas = df_final.shape[0]
colunas = df_final.shape[1]
print(f"• {linhas:,} registros")
print(f"• {colunas} colunas")

# 2. Total de municípios únicos (Corrigido para 'Município')
municipios_unicos = df_final['Município'].nunique()
print(f"• {municipios_unicos:,} municípios")

# 3. Total de Unidades da Federação (UFs)
ufs_unicas = df_final['uf_sigla'].nunique()
print(f"• {ufs_unicas} Unidades da Federação")

# 4. Verificação de valores nulos em toda a base
nulos_totais = df_final.isnull().sum().sum()
if nulos_totais == 0:
    print("• Nenhum valor nulo")
else:
    print(f"• Atenção: Existem {nulos_totais} valores nulos na base!")

# 5. Verificação de duplicados (Corrigido para 'Município')
colunas_duplicadas = ['Município', 'sexo', 'cor_raca', 'Idade']
total_duplicados = df_final.duplicated(subset=colunas_duplicadas).sum()

if total_duplicados == 0:
    print("• Nenhum registro duplicado considerando município, sexo, cor ou raça e grupo de idade")
else:
    print(f"• Atenção: Foram encontrados {total_duplicados} registros duplicados de perfil!")


====== 1. VISÃO GERAL DA BASE ANALISADA ======

• 20,709 registros
• 21 colunas
• 5,569 municípios
• 27 Unidades da Federação
• Nenhum valor nulo
• Nenhum registro duplicado considerando município, sexo, cor ou raça e grupo de idade
